In [1]:
import sys
import os
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 确保 src/ 包可被导入
root_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, root_dir)

from config import COAL_TYPES, TRAIN_DIR, TEST_DIR, AUX_COLS, ALPHAS
from src.data   import load_labels, load_coal_spectra
from src.submit import pack_submission

In [6]:
label_map, aux_map = load_labels()
coal_type = COAL_TYPES[0]
train_data = load_coal_spectra(TRAIN_DIR, coal_type, label_map, aux_map)
y          = train_data['targets']
aux        = train_data['aux']
n_batches = train_data['n_batches']
groups  = train_data['groups']

## PLS2 on raw spectral data

We would first like to try how PLS2 perform in terms of prediction auxiliary variables using the raw spectral data.

In [20]:
from src.features import compute_features

inorm = compute_features(train_data)

In [8]:
inorm.shape, aux.shape

((140, 7305), (140, 4))

In [7]:
from src.model import get_cv_splits
splits = get_cv_splits(groups, n_batches)

In [49]:
from sklearn.cross_decomposition import PLSRegression

def pls_as_predictor(k):
    oof = np.zeros_like(aux)

    for tr_idx, val_idx in splits:
        # PLS2 handles multi-output targets (n_shots, 4) natively
        pls2 = PLSRegression(n_components=k)
        # Fit PLS2 on ALL auxiliary targets simultaneously using training fold
        pls2.fit(inorm[tr_idx], aux[tr_idx])
        
        # Predict multi-output targets for validation fold
        oof[val_idx, :] = pls2.predict(inorm[val_idx])
    
    return oof

In [12]:
rmse = np.sqrt(np.mean(((oof - aux) ** 2), axis=0))
for idx in range(len(rmse)):
    print(f"Aux variable {AUX_COLS[idx]}")
    print(f"  RMSE: {rmse[idx]: .4f}")
    print(f"  Percentage error: {rmse[idx] / np.mean(aux[:,idx]) * 100: .2f}%")

Aux variable 全水分
  RMSE:  1.4599
  Percentage error:  15.69%
Aux variable 灰分
  RMSE:  3.5812
  Percentage error:  16.44%
Aux variable 氢
  RMSE:  0.0784
  Percentage error:  3.45%
Aux variable 硫
  RMSE:  0.0612
  Percentage error:  13.94%


Not an improvement, but we will try to select the optimal number of components.

In [51]:
MAX_COMPONENTS = 15
min_rmse = float('inf')
best_k = None

for k in range(MAX_COMPONENTS):
    num_component = k + 1

    oof = pls_as_predictor(num_component)

    rmse = np.sqrt(np.mean(((oof - aux) ** 2), axis=0))
    per_error = rmse / np.mean(aux, axis=0)
    
    if np.mean(per_error).item() < min_rmse:
        min_rmse = np.mean(per_error).item()
        best_k = num_component

    print(f"Evaluation for n_coponents = {num_component}")
    print("---")
    for idx in range(len(rmse)):
        print(f"Aux variable {AUX_COLS[idx]} | RMSE: {rmse[idx]: .3f} | Percentage error: {per_error[idx] * 100: .2f}%")
    print("---")
    print(f"Overall error rate: {np.mean(per_error) * 100: .2f}%\n")

print(f"The optimal number of components is: {best_k}")

Evaluation for n_coponents = 1
---
Aux variable 全水分 | RMSE:  1.369 | Percentage error:  14.72%
Aux variable 灰分 | RMSE:  4.102 | Percentage error:  18.83%
Aux variable 氢 | RMSE:  0.090 | Percentage error:  3.97%
Aux variable 硫 | RMSE:  0.057 | Percentage error:  12.88%
---
Overall error rate:  12.60%

Evaluation for n_coponents = 2
---
Aux variable 全水分 | RMSE:  1.401 | Percentage error:  15.06%
Aux variable 灰分 | RMSE:  4.267 | Percentage error:  19.58%
Aux variable 氢 | RMSE:  0.094 | Percentage error:  4.12%
Aux variable 硫 | RMSE:  0.056 | Percentage error:  12.73%
---
Overall error rate:  12.87%

Evaluation for n_coponents = 3
---
Aux variable 全水分 | RMSE:  1.392 | Percentage error:  14.96%
Aux variable 灰分 | RMSE:  4.228 | Percentage error:  19.40%
Aux variable 氢 | RMSE:  0.094 | Percentage error:  4.15%
Aux variable 硫 | RMSE:  0.057 | Percentage error:  12.95%
---
Overall error rate:  12.87%

Evaluation for n_coponents = 4
---
Aux variable 全水分 | RMSE:  1.343 | Percentage error:  14.44%

It seems like n = 8 gives the best prediction, but the error rate is still not very different from the baseline.

## PLS as a feature constructor

We now use PLS as a feature constructor and try to predict the auxiliary variables together with hand crafted features using Ridge.

In [35]:
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
import numpy as np

def pls_as_reducer(k):
    oof = np.zeros_like(aux)
    hand_feats = np.hstack([train_data['stats'], train_data['labs'], 
                            train_data['lrel'], train_data['rats']])
    hand_feats = np.nan_to_num(hand_feats, nan=0.0, posinf=0.0, neginf=0.0)

    for tr_idx, val_idx in splits:
        X_tr, aux_tr = inorm[tr_idx], aux[tr_idx]
        X_val = inorm[val_idx]

        # 1. Fit PLS2 on ALL auxiliary targets using ONLY the training fold
        pls2 = PLSRegression(n_components=k)
        T_tr, _ = pls2.fit_transform(X_tr, aux_tr)
        T_val = pls2.transform(X_val)

        # print("T_tr HAS INF/NAN:", np.isnan(T_tr).any() or np.isinf(T_tr).any())
        # print("T_val HAS INF/NAN:", np.isnan(T_val).any() or np.isinf(T_val).any())
        # print("hand_feats HAS INF/NAN:", np.isnan(hand_feats).any() or np.isinf(hand_feats).any())
        # print("T_tr MAX VALUE:", np.nanmax(np.abs(T_tr)))
        
        # 2. Scale ONLY handcrafted features (Fit on Train, Transform on Val)
        scaler_hand = StandardScaler()
        hand_tr_scaled = scaler_hand.fit_transform(hand_feats[tr_idx])
        hand_val_scaled = scaler_hand.transform(hand_feats[val_idx])  # Correct: .transform() only!

        # 3. Combine unscaled PLS scores with scaled handcrafted features
        X_tr_full = np.hstack([T_tr, hand_tr_scaled])
        X_val_full = np.hstack([T_val, hand_val_scaled])

        # 4. Fit RidgeCV to predict each auxiliary target
        for col_idx in range(len(AUX_COLS)):
            y_aux = aux_tr[:, col_idx]
            ridge = RidgeCV(alphas=ALPHAS)
            ridge.fit(X_tr_full, y_aux)
            oof[val_idx, col_idx] = ridge.predict(X_val_full)

    return oof

In [47]:
MAX_COMPONENTS = 15
min_rmse = float('inf')
best_k = None

for k in range(MAX_COMPONENTS):
    num_component = k + 1

    oof = pls_as_reducer(num_component)

    rmse = np.sqrt(np.mean(((oof - aux) ** 2), axis=0))
    per_error = rmse / np.mean(aux, axis=0)

    if np.mean(per_error).item() < min_rmse:
        min_rmse = np.mean(per_error).item()
        best_k = num_component

    print(f"Evaluation for n_coponents = {num_component}")
    print("---")
    for idx in range(len(rmse)):
        print(f"Aux variable {AUX_COLS[idx]} | RMSE: {rmse[idx]: .3f} | Percentage error: {per_error[idx] * 100: .2f}%")
    print("---")
    print(f"Overall error rate: {np.mean(per_error) * 100: .2f}%\n")

print(f"The optimal number of components is: {best_k}")

Evaluation for n_coponents = 1
---
Aux variable 全水分 | RMSE:  1.475 | Percentage error:  15.86%
Aux variable 灰分 | RMSE:  4.468 | Percentage error:  20.51%
Aux variable 氢 | RMSE:  0.098 | Percentage error:  4.31%
Aux variable 硫 | RMSE:  0.057 | Percentage error:  12.93%
---
Overall error rate:  13.40%

Evaluation for n_coponents = 2
---
Aux variable 全水分 | RMSE:  1.478 | Percentage error:  15.89%
Aux variable 灰分 | RMSE:  4.391 | Percentage error:  20.16%
Aux variable 氢 | RMSE:  0.096 | Percentage error:  4.22%
Aux variable 硫 | RMSE:  0.059 | Percentage error:  13.35%
---
Overall error rate:  13.41%

Evaluation for n_coponents = 3
---
Aux variable 全水分 | RMSE:  1.480 | Percentage error:  15.91%
Aux variable 灰分 | RMSE:  4.024 | Percentage error:  18.47%
Aux variable 氢 | RMSE:  0.087 | Percentage error:  3.84%
Aux variable 硫 | RMSE:  0.060 | Percentage error:  13.61%
---
Overall error rate:  12.96%

Evaluation for n_coponents = 4
---
Aux variable 全水分 | RMSE:  1.439 | Percentage error:  15.47%

## Predicting the Stage 2

In [52]:
from src.features import build_feature_matrix

X_spec, scaler_spec, pca, scaler_hand = build_feature_matrix(
        train_data, n_batches, fit=True)

In [ ]:
# PLS as a predictor of the auxiliary variables
oof_pls_predictor = pls_as_predictor(k = 9) # We choose the base parameter
oof_pls_predictor.shape

(140, 4)

In [65]:
X_s2 = np.hstack([X_spec, oof_pls_predictor])
scaler_s2 = StandardScaler()
#X_s2      = scaler_s2.fit_transform(np.nan_to_num(X_s2))

In [57]:
def stage2(X_s2):
    oof_batch_preds, oof_batch_true, batch_rmses = [], [], []

    for tr_idx, val_idx in splits:
        m2 = RidgeCV(alphas=ALPHAS)
        m2.fit(X_s2[tr_idx], y[tr_idx])
        val_pred   = m2.predict(X_s2[val_idx])
        val_groups = groups[val_idx]

        fold_se = []
        for bg in np.unique(val_groups):
            mask   = val_groups == bg
            true_q = float(y[val_idx][mask][0])
            pred_q = float(np.median(val_pred[mask]))
            oof_batch_preds.append(pred_q)
            oof_batch_true.append(true_q)
            fold_se.append((true_q - pred_q) ** 2)
        batch_rmses.append(float(np.sqrt(np.mean(fold_se))))

    cv_rmse_raw = float(np.mean(batch_rmses))
    print(f"Average RMSE across the batch: {cv_rmse_raw: .2f}")

In [66]:
stage2(X_s2)

Average RMSE across the batch:  354.61


In [60]:
# PLS as a a reducer
oof_pls_reducer = pls_as_reducer(k = 9) # We choose the base parameter
oof_pls_reducer.shape

(140, 4)

In [67]:
X_s2_reducer = np.hstack([X_spec, oof_pls_reducer])
scaler_s2_reducer = StandardScaler()
#X_s2_reducer= scaler_s2_reducer.fit_transform(np.nan_to_num(X_s2_reducer))

In [68]:
stage2(X_s2_reducer)

Average RMSE across the batch:  322.64
